In [20]:
import os
import numpy as np

In [21]:
data_dir = os.path.join("..", "..", "data")
os.getcwd()

'/home/ubuntu/sillicon-retina/src/prototyping'

In [22]:
print(os.listdir(data_dir))

['saliency', 'visualization_batch.zip', 'events']


# Visualize saliency

In [23]:
saliency_path = os.path.join(data_dir, "saliency")
datapoints = sorted(os.listdir(saliency_path))

idx = 6

datapoint_path = os.path.join(saliency_path, datapoints[idx])

for path in os.listdir(datapoint_path):
    print("- ",path)
    
raw_images = np.load(os.path.join(datapoint_path, "raw_images.npz.npy"))
print("raw_images : ",raw_images.shape)

retina_grids= np.load(os.path.join(datapoint_path, "cache_B=5_retina_grids.npy"))
print("retina_grids : ",retina_grids.shape)

retina_outputs = np.load(os.path.join(datapoint_path, "cache_B=5_retina_outputs.npy"))
print("retina_outputs : ",retina_outputs.shape)


-  video_upsampled_fps_eq_200.mp4
-  outputs.npz
-  video_upsampled_using_optical_flow.mp4
-  raw_images.npz.npy
-  params.json
-  cache_B=5_retina_grids.npy
-  cache_B=5_retina_outputs.npy
-  cache_B=5_ctn=0p2000_ctp=0p2000_v2_grids.npy
-  raw_images.npz
-  cache_B=5_ctn=0p2000_ctp=0p2000_v2_outputs.npy
raw_images :  (429, 180, 320, 3)
retina_grids :  (428, 5, 180, 320)
retina_outputs :  (429, 180, 320, 3)


In [24]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from ipywidgets import interact, IntSlider

# --- 1. Improved Helper Functions ---

def get_event_frame(grid_tensor):
    """
    Collapses a (B, H, W) voxel grid into a single 2D visualization.
    """
    # Custom colormap: Blue (neg) -> Grey (zero) -> Red (pos)
    colors = [(0, 0, 1), (0.5, 0.5, 0.5), (1, 0, 0)]
    cmap = mcolors.LinearSegmentedColormap.from_list("EventMap", colors)
    
    # Sum over temporal bins (B)
    flat_img = np.sum(grid_tensor, axis=0)
    
    # Robust normalization (clip top 1% outliers)
    limit = np.percentile(np.abs(flat_img), 99)
    if limit == 0: limit = 1
    
    return flat_img, cmap, limit

def overlay_saliency(bg_image, saliency_map, alpha=0.75, cmap_name='jet', grayscale_bg=True, gamma=1.0):
    """
    Overlays a saliency map with improved visibility options.
    
    Args:
        bg_image: (H, W, 3) RGB image.
        saliency_map: (H, W) or (H, W, 3) Saliency prediction.
        alpha: Max opacity of the heatmap (0.0 to 1.0).
        grayscale_bg: If True, converts background to B&W to make heatmap pop.
        gamma: Power law transform. <1.0 boosts faint regions, >1.0 sharpens peaks.
    """
    # 1. Normalize Background
    if bg_image.max() > 1.0:
        bg_image = bg_image / 255.0
        
    # Optional: Convert BG to grayscale to make colors pop
    if grayscale_bg:
        # Simple luminosity method
        gray = 0.299*bg_image[...,0] + 0.587*bg_image[...,1] + 0.114*bg_image[...,2]
        bg_image = np.stack([gray, gray, gray], axis=-1)

    # 2. Process Saliency Map
    if len(saliency_map.shape) == 3:
        saliency_map = np.mean(saliency_map, axis=-1)
        
    # Normalize to [0, 1]
    s_min, s_max = saliency_map.min(), saliency_map.max()
    if s_max - s_min > 0:
        s_norm = (saliency_map - s_min) / (s_max - s_min)
    else:
        s_norm = saliency_map
    
    # 3. Gamma Correction (Visualization Trick)
    # Raising to a power helps visibility. 
    # Use gamma < 1 (e.g. 0.5) to see wider areas, gamma > 1 to see only peaks.
    s_norm = np.power(s_norm, gamma)

    # 4. Apply Colormap
    cmap = plt.get_cmap(cmap_name)
    heatmap = cmap(s_norm)[..., :3]  # Get RGB, drop Alpha
    
    # 5. Weighted Blending
    # We weight the heatmap opacity by the saliency value itself.
    # Low saliency = transparent. High saliency = opaque (up to 'alpha').
    heatmap_opacity = s_norm[..., None] * alpha
    
    overlay = (1 - heatmap_opacity) * bg_image + heatmap_opacity * heatmap
    
    return np.clip(overlay, 0, 1)

# --- 2. Interactive Browser ---

def show_frame(idx):
    print(f"Viewing Frame Index: {idx}")
    
    # Load data
    raw_img = raw_images[idx]
    grid = retina_grids[idx]
    saliency_out = retina_outputs[idx]
    
    # Setup Figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # A. Original Image
    axes[0].imshow(raw_img)
    axes[0].set_title("Input: RGB Frame")
    axes[0].axis('off')
    
    # B. Event Representation
    ev_img, ev_cmap, ev_limit = get_event_frame(grid)
    axes[1].imshow(ev_img, cmap=ev_cmap, vmin=-ev_limit, vmax=ev_limit)
    axes[1].set_title("Event Grid (Accumulated)")
    axes[1].axis('off')
    
    # C. Saliency Heatmap
    # Set grayscale_bg=True to make the Red/Blue heatmap highly visible
    heatmap_vis = overlay_saliency(raw_img, saliency_out, 
                                   alpha=0.8, 
                                   grayscale_bg=True,  # Key change for visibility
                                   cmap_name='jet') 
    axes[2].imshow(heatmap_vis)
    axes[2].set_title("Prediction: Saliency Heatmap")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# --- 3. Run ---
interact(show_frame, idx=IntSlider(min=0, max=len(raw_images)-1, step=1));

interactive(children=(IntSlider(value=0, description='idx', max=428), Output()), _dom_classes=('widget-interac…

In [25]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Improved Export Logic ---

def save_publication_figures(idx, raw_images, retina_grids, retina_outputs, output_dir="figures"):
    """
    Saves individual components of a specific frame for publication stitching.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 1. Prepare Data
    raw_img = raw_images[idx]
    grid = retina_grids[idx]
    saliency_out = retina_outputs[idx]

    # Ensure RGB is [0, 1] for saving
    if raw_img.max() > 1.0:
        raw_img = raw_img / 255.0

    # 2. Save Original RGB
    plt.imsave(os.path.join(output_dir, f"frame_{idx}_rgb.png"), raw_img)

    # 3. Save Event Representation (Spatio-Temporal Voxel Grid)
    # Following Section 3.2 logic: sum polarities over B bins 
    ev_img, ev_cmap, ev_limit = get_event_frame(grid)
    plt.imsave(os.path.join(output_dir, f"frame_{idx}_events.png"), 
               ev_img, cmap=ev_cmap, vmin=-ev_limit, vmax=ev_limit)

    # 4. Save Saliency Heatmap (Grayscale BG for maximum visibility)
    # We use the improved overlay logic to ensure the saliency pops
    heatmap_vis = overlay_saliency(raw_img, saliency_out, 
                                   alpha=0.85, 
                                   grayscale_bg=True, 
                                   gamma=0.8, # Slightly boost faint areas
                                   cmap_name='jet')
    plt.imsave(os.path.join(output_dir, f"frame_{idx}_saliency.png"), heatmap_vis)

    print(f"Success: Figures for frame {idx} saved to '{output_dir}/'")

# --- Technical Implementation Details ---

def get_event_frame(grid_tensor):
    """
    Visualizes the event stream as a 2D image by summing interpolated polarities 
    across B bins[cite: 125, 130].
    """
    colors = [(0, 0, 1), (0.5, 0.5, 0.5), (1, 0, 0)] # Blue (Neg) -> Grey -> Red (Pos)
    cmap = mcolors.LinearSegmentedColormap.from_list("EventMap", colors)
    
    flat_img = np.sum(grid_tensor, axis=0)
    limit = np.percentile(np.abs(flat_img), 99)
    if limit == 0: limit = 1
    
    return flat_img, cmap, limit

def overlay_saliency(bg_image, saliency_map, alpha=0.75, cmap_name='jet', grayscale_bg=True, gamma=1.0):
    """
    Advanced heatmap overlay. Converts background to grayscale to emphasize 
    bio-plausible saliency detections.
    """
    # Normalize Background
    if bg_image.max() > 1.0: bg_image = bg_image / 255.0
        
    if grayscale_bg:
        gray = 0.299*bg_image[...,0] + 0.587*bg_image[...,1] + 0.114*bg_image[...,2]
        bg_image = np.stack([gray, gray, gray], axis=-1)

    # Process Saliency
    if len(saliency_map.shape) == 3:
        saliency_map = np.mean(saliency_map, axis=-1)
        
    s_norm = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min() + 1e-8)
    s_norm = np.power(s_norm, gamma)

    cmap = plt.get_cmap(cmap_name)
    heatmap = cmap(s_norm)[..., :3] 
    
    heatmap_opacity = s_norm[..., None] * alpha
    overlay = (1 - heatmap_opacity) * bg_image + heatmap_opacity * heatmap
    
    return np.clip(overlay, 0, 1)

# --- EXECUTION ---
# Replace '100' with the index you selected from the slider
selected_idx = 300
save_publication_figures(selected_idx, raw_images, retina_grids, retina_outputs)

Success: Figures for frame 300 saved to 'figures/'


In [26]:
import os
import numpy as np
import imageio as iio
import matplotlib.pyplot as plt

def save_the_video(output_name, raw_images, retina_grids, retina_outputs, output_dir="figures"):
    os.makedirs(output_dir, exist_ok=True)

    if not output_name.endswith(".mp4"):
        output_name += ".mp4"

    video_path = os.path.join(output_dir, output_name)

    with iio.get_writer(video_path, fps=20) as writer:
        for idx in range(len(retina_grids)):
            raw_img = raw_images[idx]
            grid = retina_grids[idx]
            saliency_out = retina_outputs[idx]

            # ---- Normalize RGB input ----
            if raw_img.max() > 1.0:
                raw_img = raw_img / 255.0
            raw_rgb = np.clip(raw_img, 0, 1)

            # ---- Event frame ----
            ev_img, ev_cmap, ev_limit = get_event_frame(grid)

            # Normalize events to [0,1]
            ev_norm = (ev_img + ev_limit) / (2 * ev_limit)
            ev_norm = np.clip(ev_norm, 0, 1)

            # Convert to RGB via colormap
            ev_rgb = plt.get_cmap(ev_cmap)(ev_norm)[..., :3]

            # ---- Saliency overlay ----
            saliency_rgb = overlay_saliency(
                raw_rgb,
                saliency_out,
                alpha=0.85,
                grayscale_bg=True,
                gamma=0.8,
                cmap_name="jet",
            )

            # ---- Sanity: enforce identical spatial size ----
            H, W, _ = raw_rgb.shape
            assert ev_rgb.shape[:2] == (H, W)
            assert saliency_rgb.shape[:2] == (H, W)

            # ---- Concatenate vertically (axis=0) ----
            video_frame = np.concatenate(
                [raw_rgb, ev_rgb, saliency_rgb],
                axis=1
            )

            # ---- Write frame ----
            frame_uint8 = (video_frame * 255).astype(np.uint8)
            writer.append_data(frame_uint8)

    print(f"Success: concatenated video saved to '{video_path}'")


save_the_video("saliency_vis_retina.mp4", raw_images, retina_grids, retina_outputs)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (960, 180) to (960, 192) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Success: concatenated video saved to 'figures/saliency_vis_retina.mp4'


# Visualize the DVS\

In [27]:
saliency_path = os.path.join(data_dir, "saliency")
datapoints = sorted(os.listdir(saliency_path))

idx = 6

datapoint_path = os.path.join(saliency_path, datapoints[idx])

for path in os.listdir(datapoint_path):
    print("- ",path)
    
raw_images = np.load(os.path.join(datapoint_path, "raw_images.npz.npy"))
print("raw_images : ",raw_images.shape)

retina_grids= np.load(os.path.join(datapoint_path, "cache_B=5_ctn=0p2000_ctp=0p2000_v2_grids.npy"))
print("retina_grids : ",retina_grids.shape)

retina_outputs = np.load(os.path.join(datapoint_path, "cache_B=5_ctn=0p2000_ctp=0p2000_v2_outputs.npy"))
print("retina_outputs : ",retina_outputs.shape)


-  video_upsampled_fps_eq_200.mp4
-  outputs.npz
-  video_upsampled_using_optical_flow.mp4
-  raw_images.npz.npy
-  params.json
-  cache_B=5_retina_grids.npy
-  cache_B=5_retina_outputs.npy
-  cache_B=5_ctn=0p2000_ctp=0p2000_v2_grids.npy
-  raw_images.npz
-  cache_B=5_ctn=0p2000_ctp=0p2000_v2_outputs.npy
raw_images :  (429, 180, 320, 3)
retina_grids :  (428, 5, 180, 320)
retina_outputs :  (429, 180, 320)


In [28]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from ipywidgets import interact, IntSlider

# --- 1. Improved Helper Functions ---

def get_event_frame(grid_tensor):
    """
    Collapses a (B, H, W) voxel grid into a single 2D visualization.
    """
    # Custom colormap: Blue (neg) -> Grey (zero) -> Red (pos)
    colors = [(0, 0, 1), (0.5, 0.5, 0.5), (1, 0, 0)]
    cmap = mcolors.LinearSegmentedColormap.from_list("EventMap", colors)
    
    # Sum over temporal bins (B)
    flat_img = np.sum(grid_tensor, axis=0)
    
    # Robust normalization (clip top 1% outliers)
    limit = np.percentile(np.abs(flat_img), 99)
    if limit == 0: limit = 1
    
    return flat_img, cmap, limit

def overlay_saliency(bg_image, saliency_map, alpha=0.75, cmap_name='jet', grayscale_bg=True, gamma=1.0):
    """
    Overlays a saliency map with improved visibility options.
    
    Args:
        bg_image: (H, W, 3) RGB image.
        saliency_map: (H, W) or (H, W, 3) Saliency prediction.
        alpha: Max opacity of the heatmap (0.0 to 1.0).
        grayscale_bg: If True, converts background to B&W to make heatmap pop.
        gamma: Power law transform. <1.0 boosts faint regions, >1.0 sharpens peaks.
    """
    # 1. Normalize Background
    if bg_image.max() > 1.0:
        bg_image = bg_image / 255.0
        
    # Optional: Convert BG to grayscale to make colors pop
    if grayscale_bg:
        # Simple luminosity method
        gray = 0.299*bg_image[...,0] + 0.587*bg_image[...,1] + 0.114*bg_image[...,2]
        bg_image = np.stack([gray, gray, gray], axis=-1)

    # 2. Process Saliency Map
    if len(saliency_map.shape) == 3:
        saliency_map = np.mean(saliency_map, axis=-1)
        
    # Normalize to [0, 1]
    s_min, s_max = saliency_map.min(), saliency_map.max()
    if s_max - s_min > 0:
        s_norm = (saliency_map - s_min) / (s_max - s_min)
    else:
        s_norm = saliency_map
    
    # 3. Gamma Correction (Visualization Trick)
    # Raising to a power helps visibility. 
    # Use gamma < 1 (e.g. 0.5) to see wider areas, gamma > 1 to see only peaks.
    s_norm = np.power(s_norm, gamma)

    # 4. Apply Colormap
    cmap = plt.get_cmap(cmap_name)
    heatmap = cmap(s_norm)[..., :3]  # Get RGB, drop Alpha
    
    # 5. Weighted Blending
    # We weight the heatmap opacity by the saliency value itself.
    # Low saliency = transparent. High saliency = opaque (up to 'alpha').
    heatmap_opacity = s_norm[..., None] * alpha
    
    overlay = (1 - heatmap_opacity) * bg_image + heatmap_opacity * heatmap
    
    return np.clip(overlay, 0, 1)

# --- 2. Interactive Browser ---

def show_frame(idx):
    print(f"Viewing Frame Index: {idx}")
    
    # Load data
    raw_img = raw_images[idx]
    grid = retina_grids[idx]
    saliency_out = retina_outputs[idx]
    
    # Setup Figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # A. Original Image
    axes[0].imshow(raw_img)
    axes[0].set_title("Input: RGB Frame")
    axes[0].axis('off')
    
    # B. Event Representation
    ev_img, ev_cmap, ev_limit = get_event_frame(grid)
    axes[1].imshow(ev_img, cmap=ev_cmap, vmin=-ev_limit, vmax=ev_limit)
    axes[1].set_title("Event Grid (Accumulated)")
    axes[1].axis('off')
    
    # C. Saliency Heatmap
    # Set grayscale_bg=True to make the Red/Blue heatmap highly visible
    heatmap_vis = overlay_saliency(raw_img, saliency_out, 
                                   alpha=0.8, 
                                   grayscale_bg=True,  # Key change for visibility
                                   cmap_name='jet') 
    axes[2].imshow(heatmap_vis)
    axes[2].set_title("Prediction: Saliency Heatmap")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# --- 3. Run ---
interact(show_frame, idx=IntSlider(min=0, max=len(raw_images)-1, step=1));

interactive(children=(IntSlider(value=0, description='idx', max=428), Output()), _dom_classes=('widget-interac…

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Improved Export Logic ---

def save_publication_figures(idx, raw_images, retina_grids, retina_outputs, output_dir="figures"):
    """
    Saves individual components of a specific frame for publication stitching.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 1. Prepare Data
    raw_img = raw_images[idx]
    grid = retina_grids[idx]
    saliency_out = retina_outputs[idx]

    # Ensure RGB is [0, 1] for saving
    if raw_img.max() > 1.0:
        raw_img = raw_img / 255.0

    # 2. Save Original RGB
    plt.imsave(os.path.join(output_dir, f"frame_dvs_{idx}_rgb.png"), raw_img)

    # 2. Save  RGB as gray
    gray = 0.299*raw_img[...,0] + 0.587*raw_img[...,1] + 0.114*raw_img[...,2]
    plt.imsave(os.path.join(output_dir, f"frame_dvs_{idx}_gray.png"), gray, cmap='gray')


    # 3. Save Event Representation (Spatio-Temporal Voxel Grid)
    # Following Section 3.2 logic: sum polarities over B bins 
    ev_img, ev_cmap, ev_limit = get_event_frame(grid)
    plt.imsave(os.path.join(output_dir, f"frame_dvs_{idx}_events.png"), 
               ev_img, cmap=ev_cmap, vmin=-ev_limit, vmax=ev_limit)

    # 4. Save Saliency Heatmap (Grayscale BG for maximum visibility)
    # We use the improved overlay logic to ensure the saliency pops
    heatmap_vis = overlay_saliency(raw_img, saliency_out, 
                                   alpha=0.85, 
                                   grayscale_bg=True, 
                                   gamma=0.8, # Slightly boost faint areas
                                   cmap_name='jet')
    plt.imsave(os.path.join(output_dir, f"frame_dvs_{idx}_saliency.png"), heatmap_vis)

    print(f"Success: Figures for frame {idx} saved to '{output_dir}/'")

# --- Technical Implementation Details ---

def get_event_frame(grid_tensor):
    """
    Visualizes the event stream as a 2D image by summing interpolated polarities 
    across B bins[cite: 125, 130].
    """
    colors = [(0, 0, 1), (0.5, 0.5, 0.5), (1, 0, 0)] # Blue (Neg) -> Grey -> Red (Pos)
    cmap = mcolors.LinearSegmentedColormap.from_list("EventMap", colors)
    
    flat_img = np.sum(grid_tensor, axis=0)
    limit = np.percentile(np.abs(flat_img), 99)
    if limit == 0: limit = 1
    
    return flat_img, cmap, limit

def overlay_saliency(bg_image, saliency_map, alpha=0.75, cmap_name='jet', grayscale_bg=True, gamma=1.0):
    """
    Advanced heatmap overlay. Converts background to grayscale to emphasize 
    bio-plausible saliency detections.
    """
    # Normalize Background
    if bg_image.max() > 1.0: bg_image = bg_image / 255.0
        
    if grayscale_bg:
        gray = 0.299*bg_image[...,0] + 0.587*bg_image[...,1] + 0.114*bg_image[...,2]
        bg_image = np.stack([gray, gray, gray], axis=-1)

    # Process Saliency
    if len(saliency_map.shape) == 3:
        saliency_map = np.mean(saliency_map, axis=-1)
        
    s_norm = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min() + 1e-8)
    s_norm = np.power(s_norm, gamma)

    cmap = plt.get_cmap(cmap_name)
    heatmap = cmap(s_norm)[..., :3] 
    
    heatmap_opacity = s_norm[..., None] * alpha
    overlay = (1 - heatmap_opacity) * bg_image + heatmap_opacity * heatmap
    
    return np.clip(overlay, 0, 1)

# --- EXECUTION ---
# Replace '100' with the index you selected from the slider

i = 300
save_publication_figures(i, raw_images, retina_grids, retina_outputs)

Success: Figures for frame 270 saved to 'figures/'
Success: Figures for frame 271 saved to 'figures/'
Success: Figures for frame 272 saved to 'figures/'
Success: Figures for frame 273 saved to 'figures/'
Success: Figures for frame 274 saved to 'figures/'
Success: Figures for frame 275 saved to 'figures/'
Success: Figures for frame 276 saved to 'figures/'
Success: Figures for frame 277 saved to 'figures/'
Success: Figures for frame 278 saved to 'figures/'
Success: Figures for frame 279 saved to 'figures/'
Success: Figures for frame 280 saved to 'figures/'
Success: Figures for frame 281 saved to 'figures/'
Success: Figures for frame 282 saved to 'figures/'
Success: Figures for frame 283 saved to 'figures/'
Success: Figures for frame 284 saved to 'figures/'
Success: Figures for frame 285 saved to 'figures/'
Success: Figures for frame 286 saved to 'figures/'
Success: Figures for frame 287 saved to 'figures/'
Success: Figures for frame 288 saved to 'figures/'
Success: Figures for frame 289 

In [30]:
import os
import numpy as np
import imageio as iio
import matplotlib.pyplot as plt

def save_the_video(output_name, raw_images, retina_grids, retina_outputs, output_dir="figures"):
    os.makedirs(output_dir, exist_ok=True)

    if not output_name.endswith(".mp4"):
        output_name += ".mp4"

    video_path = os.path.join(output_dir, output_name)

    with iio.get_writer(video_path, fps=20) as writer:
        for idx in range(len(retina_grids)):
            raw_img = raw_images[idx]
            grid = retina_grids[idx]
            saliency_out = retina_outputs[idx]

            # ---- Normalize RGB input ----
            if raw_img.max() > 1.0:
                raw_img = raw_img / 255.0
            raw_rgb = np.clip(raw_img, 0, 1)

            # ---- Event frame ----
            ev_img, ev_cmap, ev_limit = get_event_frame(grid)

            # Normalize events to [0,1]
            ev_norm = (ev_img + ev_limit) / (2 * ev_limit)
            ev_norm = np.clip(ev_norm, 0, 1)

            # Convert to RGB via colormap
            ev_rgb = plt.get_cmap(ev_cmap)(ev_norm)[..., :3]

            # ---- Saliency overlay ----
            saliency_rgb = overlay_saliency(
                raw_rgb,
                saliency_out,
                alpha=0.85,
                grayscale_bg=True,
                gamma=0.8,
                cmap_name="jet",
            )

            # ---- Sanity: enforce identical spatial size ----
            H, W, _ = raw_rgb.shape
            assert ev_rgb.shape[:2] == (H, W)
            assert saliency_rgb.shape[:2] == (H, W)

            # ---- Concatenate vertically (axis=0) ----
            video_frame = np.concatenate(
                [raw_rgb, ev_rgb, saliency_rgb],
                axis=1
            )

            # ---- Write frame ----
            frame_uint8 = (video_frame * 255).astype(np.uint8)
            writer.append_data(frame_uint8)

    print(f"Success: concatenated video saved to '{video_path}'")


save_the_video("saliency_vis_dvs.mp4", raw_images, retina_grids, retina_outputs)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (960, 180) to (960, 192) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Success: concatenated video saved to 'figures/saliency_vis_dvs.mp4'
